# 04 – Bee Haven: Gold-Tabelle (N4)

**Ziel:** Aus den bereinigten Silver-Daten **eine Gold-Tabelle je Stunde** bauen: Flugbewegungen, Stocktemperatur, Stockfeuchte, Gewicht und Außenwetter nebeneinander – die Grundlage für Dashboard, Analyse (N5) und maschinelles Lernen. Dazu die **Tageswerte** je Tag.

**Vorbild:** Kurs-Notebook 05 „Silver to Gold“ (alle Sensoren einer Stadt zu einer Tabelle zusammenführen). **Abweichungen** (Pflichtenheft PH-ITS-002, Karten 3.1 und 5.9):
- **Kurze Lücken werden erst hier in Gold aufgefüllt** – Silver bleibt die unveränderte Messung. Jede aufgefüllte Spalte bekommt eine Kennzeichen-Spalte `<spalte>_aufgefuellt` (True = geschätzt).
- **flow wird nie aufgefüllt** (Zählwerte lassen sich nicht aus Nachbarminuten ableiten).
- Nur Werte mit Qualität „ok…“ (Regeln R5–R7, R5a); auffällige Flugwerte (R9) zählen nicht mit.

**Einfach erklärt:** Silver ist das Kassenbuch (nur echte Messungen), Gold der Monatsbericht (übersichtlich zusammengefasst, Schätzungen gekennzeichnet).

**Läuft:** täglich in der Pipeline (Aktivität `n4_gold`); Parameter `STADT`.

**Ergebnistexte:** gelten für jeden Standort; die Zahlen stehen in der Ausgabe. „Kontrollwerte“ = erwartete Werte je Stadt (Stand 02.10.2026).

## 0. Bibliotheken und Pfade

In [1]:
# --- Parameter: Standort (neue Stadt = Zeile in lakehouse/konfiguration/standorte.csv + Ordner in silver/processing) ---
STADT = "schwartau"                                       # Standort = Unterordner in bronze/silver/gold und Teil der Dateinamen

# --- Bibliotheken laden ---
import numpy as np                       # numpy: Rechnen mit Zahlenreihen
import pandas as pd                      # pandas: Tabellen zusammenführen und speichern
from pathlib import Path                 # Path: Ordner- und Dateipfade

# --- Pfade festlegen ---
PROJEKT = Path.cwd().parent                                                        # Projektordner
CLEANED = PROJEKT / "02_daten_ITS-Data" / "lakehouse" / "silver" / "cleaned" / STADT  # Eingang: bereinigte Daten
GOLD = PROJEKT / "02_daten_ITS-Data" / "lakehouse" / "gold" / STADT                         # Ausgang: Gold-Tabellen
sensoren = ["flow", "humidity", "temperature", "weight"]                           # Liste der Sensoren

# --- Standortdaten aus der Konfiguration (Koordinaten, Zeitzone) ---
standorte = pd.read_csv(PROJEKT / "02_daten_ITS-Data" / "lakehouse" / "konfiguration" / "standorte.csv", index_col="stadt")
if STADT not in standorte.index:                          # unbekannte Stadt -> sofort abbrechen statt mit falschen Daten weiterrechnen
    raise ValueError(f"Standort '{STADT}' fehlt in konfiguration/standorte.csv – Zeile mit Koordinaten und Zeitzone ergänzen")
ORT = standorte.loc[STADT]                                # eine Zeile: name, breite, laenge, zeitzone, quelle

# --- Sensor-Regeln je Standort (Einheit der Rohdaten, gültiger Bereich) und bekannte Ausfallzeiträume ---
KONFIG = PROJEKT / "02_daten_ITS-Data" / "lakehouse" / "konfiguration"                  # Ordner mit den Konfigurationstabellen
sensoren_cfg = pd.read_csv(KONFIG / "sensoren.csv")                                    # eine Zeile je Stadt und Sensor
REGELN = sensoren_cfg[sensoren_cfg["stadt"] == STADT].set_index("sensor")              # Regeln dieser Stadt: einheit_roh, min, max
ausschluesse = pd.read_csv(KONFIG / "ausschluesse.csv")                                # bekannte Ausfallzeiträume (z. B. Stock nicht auf Waage)
AUSSCHLUESSE = ausschluesse[ausschluesse["stadt"] == STADT].copy()                     # nur Zeiträume dieser Stadt
AUSSCHLUESSE[["von_utc", "bis_utc"]] = AUSSCHLUESSE[["von_utc", "bis_utc"]].apply(pd.to_datetime, utc=True)  # Text -> UTC-Zeit
ZEITZONE = ORT["zeitzone"]                                # Ortszeit der Rohdaten (Deutschland: Europe/Berlin)
print("Standort:", STADT, "·", ORT["name"], "· Zeitzone", ZEITZONE)

Standort: schwartau · Bad Schwartau · Zeitzone Europe/Berlin


**Ergebnis:** Standort und Pfade gesetzt.

## 1. Neueste bereinigte Version einlesen

Je Sensor die jüngste Datei aus `silver/cleaned/<stadt>/<sensor>/` (Name enthält den UTC-Zeitstempel), dazu Wetter (stündlich) und Sonnenzeiten (täglich) aus N3.

In [2]:
# --- Je Sensor die neueste Datei in cleaned finden und einlesen ---
daten = {}                                                              # leeres Dictionary: Sensor -> Tabelle
for s in sensoren:                                                      # für jeden Sensor ...
    dateien = sorted((CLEANED / s).glob(f"{s}_{STADT}_cleaned__*.csv"))  # ... alle Versionen suchen (Name enthält Zeitstempel)
    neueste = dateien[-1]                                               # ... letzte nach Namen = neueste Version
    daten[s] = pd.read_csv(neueste)                                     # ... einlesen
    daten[s]["zeit_utc"] = pd.to_datetime(daten[s]["zeit_utc"], utc=True)  # ... CSV liefert Text -> wieder in UTC-Zeit umwandeln
    print(f"{s:12} {len(daten[s]):>10,} Zeilen  aus {neueste.name}  ({len(dateien)} Version(en) vorhanden)")  # ... Kontrolle

flow          1,256,918 Zeilen  aus flow_schwartau_cleaned__2026-10-01_151930_UTC.csv  (1 Version(en) vorhanden)
humidity          1,761 Zeilen  aus humidity_schwartau_cleaned__2026-10-01_151930_UTC.csv  (1 Version(en) vorhanden)


temperature     253,430 Zeilen  aus temperature_schwartau_cleaned__2026-10-01_151930_UTC.csv  (1 Version(en) vorhanden)
weight            1,761 Zeilen  aus weight_schwartau_cleaned__2026-10-01_151930_UTC.csv  (1 Version(en) vorhanden)


In [3]:
# --- Neueste Wetter- und Sonnendaten aus N3 laden ---
neueste = lambda ordner: sorted((CLEANED / ordner).glob(f"{ordner}_{STADT}_cleaned__*.csv"))[-1]   # jüngste Version
wetter = pd.read_csv(neueste("wetter"), parse_dates=["zeit_utc"])                                   # stündlich
sonne = pd.read_csv(neueste("sonne"), parse_dates=["daemmerung_beginn_utc", "sonnenaufgang_utc",
                                                    "sonnenuntergang_utc", "daemmerung_ende_utc"])   # täglich
wetter["zeit_utc"] = pd.to_datetime(wetter["zeit_utc"], utc=True)                                  # sicherstellen: UTC-Zeit
print(f"wetter {len(wetter):,} Stunden · sonne {len(sonne):,} Tage")

wetter 21,145 Stunden · sonne 881 Tage


**Ergebnis:** Zeilen je Sensor und Wetter/Sonne wie in N2/N3 (Ausgabe nennt auch die gelesene Version).

| Kontrollwerte | flow · humidity · temperature · weight · Wetter · Sonne |
| :-- | :-- |
| Schwartau | 1.256.918 · 1.761 · 253.430 · 1.761 · 21.145 h · 881 Tage |
| Würzburg | 1.027.761 · 20.845 · 958.831 · 1.035.861 · 20.881 h · 870 Tage |

## 2. Nur gültige Werte verwenden

Nur Zeilen mit Qualität „ok…“ (ungültige Werte aus R5, R5a, R6 und fehlende Werte fließen nicht ein). Bei flow zählen auffällige Werte (R9) nicht mit.

In [4]:
# --- Nur Zeilen mit Qualität "ok..." behalten ---
gueltig = {}                                                            # leeres Dictionary für die gültigen Daten
for s in sensoren:                                                      # für jeden Sensor ...
    maske = daten[s]["qualitaet"].str.startswith("ok")                  # ... True, wenn die Qualität mit "ok" beginnt
    gueltig[s] = daten[s][maske]                                        # ... nur diese Zeilen übernehmen
    print(f"{s:12} {len(gueltig[s]):>10,} gültige Zeilen")              # ... Kontrolle

flow          1,256,918 gültige Zeilen
humidity          1,718 gültige Zeilen
temperature     251,398 gültige Zeilen
weight            1,619 gültige Zeilen


**Ergebnis:** gültige Zeilen je Sensor nach den Regeln aus N2.

| Kontrollwerte | gültig: flow · humidity · temperature · weight |
| :-- | :-- |
| Schwartau | 1.256.918 · 1.718 · 251.398 · 1.619 |
| Würzburg | 1.027.761 · 17.263 · 794.250 · 1.015.006 |

## 3. Stundentabelle bauen

**Warum je Stunde?** Das Wetter liegt stündlich vor; Minuten (flow), 5-Minuten-Werte (Temperatur) und kürzere Takte werden zur Stunde zusammengefasst, seltenere Messungen (z. B. Schwartau: Feuchte und Gewicht alle 12 Stunden) stehen in ihrer Messstunde. So passen alle Quellen auf **ein gemeinsames Zeitraster**.

**Eigene Erwartung:** Anzahl Tage × 24 Stunden; Flugbewegungen bei Tag deutlich höher als nachts.

In [5]:
# --- Gemeinsames Zeitraster: jede volle Stunde (UTC) vom ersten bis zum letzten Messwert ---
start = min(daten[s]["zeit_utc"].min() for s in sensoren).floor("h")    # früheste Messung, auf volle Stunde abgerundet
ende = max(daten[s]["zeit_utc"].max() for s in sensoren).floor("h")     # späteste Messung
gold = pd.DataFrame(index=pd.date_range(start, ende, freq="h", name="zeit_utc"))  # leere Tabelle, eine Zeile je Stunde

# --- flow: Flugbewegungen je Minute (Einflug + Ausflug als Betrag), auffällige Werte (R9) ausgeschlossen ---
fl = gueltig["flow"].set_index("zeit_utc")
ein = fl["flow_ein"].where(~fl["flow_ein_auffaellig"])                  # Einflug (positiv laut Datensatz-Beschreibung)
aus = fl["flow_aus"].abs().where(~fl["flow_aus_auffaellig"])            # Ausflug (negativ gemessen -> Betrag)
je_std = pd.DataFrame({"ein": ein, "aus": aus}).resample("h")           # nach Stunden gruppieren
gold["flug_ein_summe"] = je_std["ein"].sum(min_count=1)                 # Einflüge in der Stunde
gold["flug_aus_summe"] = je_std["aus"].sum(min_count=1)                 # Ausflüge in der Stunde
gold["flug_minuten"] = je_std["ein"].count()                            # wie viele Minuten gemessen wurden (max. 60)
gold["flug_bewegungen_pro_min"] = (gold["flug_ein_summe"] + gold["flug_aus_summe"]) / gold["flug_minuten"]  # Mittel je Minute

# --- Stocktemperatur: Stundenmittel der 5-Minuten-Werte ---
gold["stock_temperatur_c"] = gueltig["temperature"].set_index("zeit_utc")["temperature"].resample("h").mean()

# --- Feuchte und Gewicht: Messwert in seiner Messstunde (00:00 und 12:00 UTC) ---
gold["stock_feuchte_pct"] = gueltig["humidity"].set_index("zeit_utc")["humidity"].resample("h").mean()
gold["gewicht_kg"] = gueltig["weight"].set_index("zeit_utc")["weight"].resample("h").mean() / 1000   # Gramm -> kg

# --- Außenwetter der Stunde (N3) anhängen ---
spalten_wetter = ["aussentemperatur_c", "niederschlag_mm", "sonnenschein_min", "wind_kmh", "windboeen_kmh",
                  "luftdruck_hpa", "taupunkt_c", "sonnenstrahlung_kwh_m2", "wetterlage", "sonnenhoehe_grad", "tageszeit"]
gold = gold.join(wetter.set_index("zeit_utc")[spalten_wetter], how="left")
gold.insert(0, "stadt", STADT)                                          # Standort als Spalte (mehrere Städte in einer Tabelle möglich)
print(f"{len(gold):,} Stunden von {gold.index.min()} bis {gold.index.max()}")
gold.notna().mean().mul(100).round(1).to_frame("gefuellt_%").T         # Kontrolle: Anteil vorhandener Werte je Spalte

21,121 Stunden von 2017-01-01 12:00:00+00:00 bis 2019-05-31 12:00:00+00:00


,stadt,flug_ein_summe,flug_aus_summe,flug_minuten,flug_bewegungen_pro_min,stock_temperatur_c,stock_feuchte_pct,gewicht_kg,aussentemperatur_c,niederschlag_mm,sonnenschein_min,wind_kmh,windboeen_kmh,luftdruck_hpa,taupunkt_c,sonnenstrahlung_kwh_m2,wetterlage,sonnenhoehe_grad,tageszeit
gefuellt_%,100.0,99.2,99.2,100.0,99.2,99.2,8.1,7.7,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,98.1,100.0,100.0


**Ergebnis:** Stunden und Zeitraum sowie der Füllgrad je Spalte (Anteil Stunden mit Wert) in der Ausgabe.

| Kontrollwerte | Stunden · Füllgrad Flug / Stocktemperatur / Feuchte / Gewicht |
| :-- | :-- |
| Schwartau | 21.121 · 99,2 % / 99,2 % / 8,1 % / 7,7 % (Feuchte und Gewicht nur alle 12 Std.) |
| Würzburg | 20.864 · 82,1 % / 82,8 % / 82,7 % / 81,1 % (Lücke Mai–Okt. 2018) |

## 4. Kurze Lücken auffüllen (Gold-Regel, Entscheidung 30.09.2026)

**Regel:** Nur **kurze** Lücken bei **gleichmäßigen** Messgrößen werden zeitlich interpoliert (linear zwischen den Nachbarwerten); jede gefüllte Zelle ist in `<spalte>_aufgefuellt` gekennzeichnet. Längere Lücken bleiben leer.

| Spalte | höchstens aufgefüllte Lücke | Warum |
| :-- | :-- | :-- |
| `stock_temperatur_c` | 2 Stunden | Temperatur im Stock ändert sich langsam |
| `stock_feuchte_pct` | 11 Stunden (zwischen zwei 12-Stunden-Messungen) | fehlt eine Messung ganz, bleibt die Lücke |
| `gewicht_kg` | 11 Stunden | wie Feuchte; Zeitraum E5 (Stock nicht auf Waage) bleibt leer |
| flow-Spalten | nie | Zählwerte, nicht schätzbar |

**Vergleich Kurs-Notebook 04:** füllt schon in Silver (Temperatur bis 60 Werte = 5 Std., Gewicht bis 8 Werte = 4 Tage) – wir erst in Gold, mit Kennzeichen und kürzeren Grenzen.

**Eigene Erwartung:** Feuchte und Gewicht werden fast vollständig stündlich; Temperatur nur wenige zusätzliche Stunden.

In [6]:
def kurze_luecken_fuellen(reihe, max_stunden):
    """Nur Lücken bis max_stunden aufeinanderfolgender fehlender Stunden zeitlich interpolieren.
    Gibt (gefüllte Reihe, Kennzeichen True = aufgefüllt) zurück; längere Lücken bleiben leer."""
    fehlt = reihe.isna()
    block = (fehlt != fehlt.shift()).cumsum()                            # Nummer je zusammenhängendem Block (fehlend / vorhanden)
    laenge = fehlt.groupby(block).transform("sum")                       # Länge des fehlenden Blocks, zu dem die Stunde gehört
    geschaetzt = reihe.interpolate(method="time", limit_area="inside")   # nur zwischen zwei echten Werten, nie am Rand
    erlaubt = fehlt & (laenge <= max_stunden) & geschaetzt.notna()       # kurze Lücke mit Nachbarwerten auf beiden Seiten
    return reihe.where(~erlaubt, geschaetzt), erlaubt

regeln = {"stock_temperatur_c": 2, "stock_feuchte_pct": 11, "gewicht_kg": 11}   # Spalte -> höchstens aufgefüllte Stunden
for spalte, max_std in regeln.items():
    gold[spalte], gold[f"{spalte}_aufgefuellt"] = kurze_luecken_fuellen(gold[spalte], max_std)
    print(f"{spalte:22} aufgefüllt: {gold[f'{spalte}_aufgefuellt'].sum():>6,} Stunden · weiter leer: {gold[spalte].isna().sum():>6,}")

stock_temperatur_c     aufgefüllt:      1 Stunden · weiter leer:    165
stock_feuchte_pct      aufgefüllt: 18,799 Stunden · weiter leer:    604
gewicht_kg             aufgefüllt: 17,765 Stunden · weiter leer:  1,737


**Ergebnis:** je Spalte aufgefüllte Stunden und Stunden, die leer bleiben (längere Ausfälle).

| Kontrollwerte | aufgefüllt / leer: Stocktemperatur · Feuchte · Gewicht |
| :-- | :-- |
| Schwartau | 1 / 165 · 18.799 / 604 · 17.765 / 1.737 (v. a. Zeitraum E5) |
| Würzburg | 0 / 3.595 · 6 / 3.595 · 7 / 3.931 (Lücke Mai–Okt. 2018) |

Bei 12-Stunden-Takten (Schwartau) macht das Auffüllen Feuchte und Gewicht fast durchgehend stündlich; bei kurzen Takten (Würzburg) gibt es kaum etwas aufzufüllen – lange Lücken bleiben in beiden Fällen leer.

## 5. Tageswerte

Je Kalendertag (UTC) Mittelwerte bzw. Summen – für Dashboard-Kacheln und Tagesvergleiche (vorher in der Analyse, jetzt hier in Gold).

In [7]:
# --- Tageswerte je Sensor berechnen ---
tag = lambda t: t.set_index("zeit_utc").resample("D")                   # Hilfsfunktion: Daten nach Kalendertag (UTC) gruppieren
gewicht = tag(gueltig["weight"])["weight"].mean() / 1000                # Tagesmittel Gewicht, Gramm -> kg
fl_ok = gueltig["flow"].assign(                                         # flow ohne auffällige Werte (R9):
    flow_aus=gueltig["flow"]["flow_aus"].where(~gueltig["flow"]["flow_aus_auffaellig"]),  #   auffällige Ausflugwerte leeren
    flow_ein=gueltig["flow"]["flow_ein"].where(~gueltig["flow"]["flow_ein_auffaellig"]))  #   auffällige Einflugwerte leeren
ausflug = -tag(fl_ok)["flow_aus"].sum()                                 # Ausflüge je Tag (negativ gemessen -> positiv)
einflug = tag(fl_ok)["flow_ein"].sum()                                  # Einflüge je Tag
temp = tag(gueltig["temperature"])["temperature"].agg(["mean", "min", "max"])  # Temperatur: Mittel, Min, Max je Tag
feuchte = tag(gueltig["humidity"])["humidity"].mean()                   # Tagesmittel Luftfeuchte

tageswerte = pd.DataFrame({                                             # alles in EINE Tabelle zusammenführen:
    "gewicht_kg": gewicht,                                              #   Gewicht
    "ausflug": ausflug, "einflug": einflug,                             #   Flugaktivität
    "flugbewegungen": ausflug + einflug,                                #   Ein- + Ausflüge
    "temp_mittel": temp["mean"], "temp_min": temp["min"], "temp_max": temp["max"],  # Temperatur
    "feuchte_mittel": feuchte,                                          #   Luftfeuchte
}).round(2)                                                             # auf 2 Nachkommastellen runden
tageswerte.insert(0, "stadt", STADT)                                    # Standort als Spalte
print(len(tageswerte), "Tage von", tageswerte.index.min().date(), "bis", tageswerte.index.max().date())  # Kontrolle
tageswerte.describe().round(1)                                          # Überblick: Min, Max, Mittelwert je Kennzahl

881 Tage von 2017-01-01 bis 2019-05-31


,gewicht_kg,ausflug,einflug,flugbewegungen,temp_mittel,temp_min,temp_max,feuchte_mittel
count,812.0,881.0,881.0,881.0,876.0,876.0,876.0,863.0
mean,60.6,30108.8,33436.5,63545.2,22.6,21.0,24.6,87.0
std,10.6,43143.2,51149.0,93957.2,11.6,12.0,11.1,12.6
min,46.6,-0.0,0.0,0.0,-5.5,-7.3,-3.7,30.9
25%,50.5,27.0,2.0,33.0,12.4,9.8,17.0,79.7
50%,60.0,4694.0,4903.0,9523.0,24.4,21.5,28.0,91.0
75%,67.7,52175.0,51281.0,104453.0,34.2,33.6,34.8,96.5
max,91.2,182808.0,233717.0,416525.0,35.4,35.1,37.5,100.0


**Ergebnis:** Tageswerte mit Zeitraum und Kennzahlen (Ausgabe).

| Kontrollwerte | Tage · Gewicht Median |
| :-- | :-- |
| Schwartau | 881 · ca. 60 kg |
| Würzburg | 870 · 53,7 kg |

## 6. Speichern nach `gold/<stadt>/`

Parquet (klein, Datentypen bleiben – Befund Z1) für Dashboard und Modelle, CSV zum Ansehen. Die alte Version wird vorher ins `_archiv` verschoben (nichts wird gelöscht).

In [8]:
# --- Gold-Tabellen speichern ---
GOLD.mkdir(parents=True, exist_ok=True)                                 # Gold-Ordner anlegen, falls nicht vorhanden
stempel_gold = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d_%H%M%S")    # Zeitpunkt, bis zu dem die alte Version galt
for name in ["stunden", "tageswerte"]:
    for alt in sorted(GOLD.glob(f"{name}_{STADT}.*")):                  # alte Version vor dem Überschreiben ins _archiv
        (GOLD / "_archiv").mkdir(exist_ok=True)
        alt.rename(GOLD / "_archiv" / f"{alt.stem}__bis_{stempel_gold}_UTC{alt.suffix}")
gold.to_parquet(GOLD / f"stunden_{STADT}.parquet")                      # Stundentabelle (Hauptergebnis)
gold.to_csv(GOLD / f"stunden_{STADT}.csv")                              # zum Ansehen
tageswerte.to_parquet(GOLD / f"tageswerte_{STADT}.parquet")            # Tageswerte
tageswerte.to_csv(GOLD / f"tageswerte_{STADT}.csv")
for datei in sorted(GOLD.glob(f"*_{STADT}.*")):                         # Kontrolle: Dateien und Größen
    print(datei.name, f"{datei.stat().st_size / 1000:.0f} KB")

stunden_schwartau.csv 3775 KB
stunden_schwartau.parquet 972 KB
tageswerte_schwartau.csv 74 KB
tageswerte_schwartau.parquet 47 KB


**Ergebnis:** Dateigrößen der Gold-Tabellen als Parquet und CSV (Parquet ca. ein Viertel der CSV-Größe, vgl. Z1).

| Kontrollwerte | stunden Parquet / CSV · tageswerte Parquet / CSV |
| :-- | :-- |
| Schwartau | 972 / 3.775 KB · 47 / 74 KB |
| Würzburg | 976 / 3.649 KB · 43 / 68 KB |

## Zusammenfassung N4

| Tabelle | Inhalt | Datei |
| :-- | :-- | :-- |
| Stunden | Flug, Stocktemperatur, Stockfeuchte, Gewicht, Außenwetter, Tageszeit; Kennzeichen `_aufgefuellt` | `gold/<stadt>/stunden_<stadt>.parquet` |
| Tageswerte | Gewicht, Flug, Temperatur, Feuchte je Tag | `gold/<stadt>/tageswerte_<stadt>.parquet` |

Nächster Schritt: N5 Analyse (bei Bedarf) und Dashboard lesen aus Gold.